
###  Column Name Standardization UDF (Silver Layer)

####  Purpose

In the Silver layer, we standardize all incoming column names to ensure consistency across different sources and ingestions.
This avoids issues like:

* Same column having different names across files
  (e.g., `Customer Name`, `customer_name`, `customerName`)
* Spaces and special characters breaking SQL queries
* Inconsistent casing (`Name` vs `name`)
---

####  What this UDF Does

This UDF takes raw column names from Bronze and converts them into a clean, uniform naming format suitable for Silver.

#### Typical standardization rules applied:

* Converts to **lowercase**
* Replaces **spaces** with underscores (`_`)
* Removes or replaces **special characters** (`-`, `.`, `#`, `(`, `)`, etc.)
* Collapses multiple underscores into a single underscore
* Trims leading/trailing underscores
* Ensures the column name is SQL-friendly and consistent
---

###  Steps Performed by the Standardization Logic

#### Step 1: Normalize case

All column names are converted to lowercase.

Example:
`Customer Name` → `customer name`

---

#### Step 2: Replace spaces with underscores

Example:
`customer name` → `customer_name`

---

#### Step 3: Remove special characters

Characters like `- . , ( ) / # % &` etc. are removed or replaced.

Example:
`order-id#` → `order_id`

---

#### Step 4: Remove duplicate underscores

Example:
`customer__name` → `customer_name`

---

#### Step 5: Trim underscores

Example:
`_customer_name_` → `customer_name`

---

###  Where This is Used in the Pipeline

This standardization is applied during **Bronze → Silver** transformation.

### Benefits

* Consistent schema across all ingestions
* Reduced schema mismatch errors
* Better maintainability for downstream transformations

---



In [0]:
CREATE OR REPLACE FUNCTION coffee.silver.standardize_column_name(col_name STRING)
RETURNS STRING
LANGUAGE PYTHON
AS $$
import re

if col_name is None:
    return None

# Convert to lowercase
name = col_name.lower()

# Replace spaces and hyphens with underscore
name = re.sub(r"[\s\-]+", "_", name)

# Remove special characters except underscore
name = re.sub(r"[^a-z0-9_]", "", name)

# Collapse multiple underscores
name = re.sub(r"_+", "_", name)

# Trim leading/trailing underscores
return name.strip("_")
$$;
